# Формирование вопроса

In [ ]:
import os
import json
import re
import tqdm

In [ ]:
with open("C:/maga/context LLM (ПИС)/bench_context/is_NPs_coref/q1-is_NPs_coref.json", 'r', encoding = 'utf-8') as f:
    q_objs = json.load(f)

In [ ]:
def obj2quest(q_obj):
    return ('Ответь на вопрос по этому тексту: "' + q_obj["text"] 
            + '"\nКореферентны ли подстроки "' + q_obj['question'][0] + '" и "' + q_obj['question'][1]
            + '"?\nНапиши только True, если кореферентны, и False, если не кореферентны, без комментариев')

def custom_bool(line): return bool(re.match(r'[Tt]rue', line))

# Запрос к модели

In [ ]:
iam_token = 't1.9euelZqKxpvKzJyTzMuTmJvMms7OjO3rnpWaj5uSm8eal5nHlMrGjMiLx83l8_c9ZlZD-e9zAHsM_t3z930UVEP573MAewz-zef1656VmseMmsjIz8vPzpSPiciZzJ6L7_zF656VmseMmsjIz8vPzpSPiciZzJ6L.gU-hI-l8zlbOU8XmrYXmo9l6iAoq8uKZp2WGM2n_Kd4tlnvWi-cwSToi2kB4ryptdQyoamICZzGtZaFTkyAcDA'
folder_id = 'b1ggrmkcatrn3fgllatd'

In [ ]:
import requests

In [ ]:
URL = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"

def run(iam_token, folder_id, user_text):
    # Собираем запрос
    data = {}
    # Указываем тип модели (https://yandex.cloud/ru/docs/foundation-models/concepts/yandexgpt/models)
    data["modelUri"] = f"gpt://{folder_id}/yandexgpt-lite"
    # Настраиваем опции
    data["completionOptions"] = {"temperature": 0.3, "maxTokens": 1000}
    # Указываем контекст для модели
    data["messages"] = [
        {"role": "system", "text": user_text},
        {"role": "user", "text": f"{user_text}"},
    ]

    # Отправляем запрос
    response = requests.post(
        URL,
        headers={
            "Accept": "application/json",
            "Authorization": f"Bearer {iam_token}"
        },
        json=data,
    ).json()

    #Распечатываем результат
    print(response)


In [ ]:
import requests

prompt = {
    "modelUri": "gpt://b1gqdais7nidpse4m2ur/yandexgpt-lite",
    "completionOptions": {
        "stream": False,
        "temperature": 0.6,
        "maxTokens": "2000"
    },
    "messages": [
        {
            "role": "user",
            "text": "К какой сущности в этом тексте реферирует слово \"тот\"? Напиши только краткий ответ в именительном падеже, без комментариев. Вот текст: Свидетельств о близости Якунина к Путину хватает. В 2005 году на пасхальной службе в храме Христа Спасителя Владимир Якунин стоял за спиной президента Путина в узком кругу приближенных\". Якунин и Путин дружили семьями и даже дачи в Ленинградской области построили рядом. В команде будущего президента Якунин работает с начала 1990-х годов.  В 1990-е годы Якунин входил в совет директоров гостиницы \"Европа\" в Санкт-Петербурге и Балтийского морского пароходства, руководил банком и работал вместе с Путиным, когда тот возглавлял комитет по внешним связям мэрии второго города России. В 1996 году он вместе с Путиным стал одним из основателей дачного кооператива \"Озеро\" под Санкт-Петербургом. В Москву он приехал в 2000 году в качестве заместителя министра транспорта и постепенно вырос до руководителя компании \"РЖД\", занявшей место министерства, отвечавшего за данный сектор."
        # },
        # {
        #     "role": "user",
        #     "text": ""
        }
    ]
}


url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
headers = {
    "Content-Type": "application/json",
    "Authorization": "Api-Key AQVN1d0qGys6kh2jsnYXF_sAPD-ljpj_ldteV0DF"
}

response = requests.post(url, headers=headers, json=prompt)
result = response.text
print(result)

## LANGCHAIN

In [ ]:
from langchain_openai import ChatOpenAI

API_KEY = ""

In [ ]:
MODEL = "gpt-3.5-turbo"
# MODEL = "gpt-4o"

model = ChatOpenAI(model = MODEL,
                   api_key = API_KEY,
                   base_url = "https://api.vsegpt.ru/v1",
                   temperature = 0)
# res = model.invoke(input())

In [ ]:
for i in range(len(q_objs)):
# for i in range(1):
    answer = model.invoke(obj2quest(q_objs[i]))
    q_objs[i]['full answer'] = str(answer)
    q_objs[i]['answer'] = answer.content
    q_objs[i]['is correct'] = custom_bool(q_objs[i]['answer']) == q_objs[i]['gold']

In [ ]:
with open(MODEL + '.json', 'w', encoding = 'utf-8') as f:
    json.dump(q_objs, f)

In [ ]:
from openai import OpenAI

API_KEY = "sk-or-vv-006f33a4cb03fa31f2863a566c71bfcaad3ad92347c725af98ada4fc9f17e8fb"

In [ ]:
MODEL = "anthropic/claude-3-haiku"

In [ ]:
# Using OpenAI API directly

client = OpenAI(
    api_key = API_KEY,
    base_url = "https://api.vsegpt.ru/v1",
)

prompt = ""

messages = []
messages.append({"role": "user", "content": input()})

response_big = client.chat.completions.create(
    model = MODEL, # id модели из списка моделей - можно использовать OpenAI, Anthropic и пр. меняя только этот параметр
    messages = messages,
    temperature = 0.7
)

response = response_big.choices[0].message.content
print("Response:", response)
